# Models 1 and 2 on a GPU — zero-shot baseline and the QLoRA fine-tune

**Linear 298-53 (Model 1) and 298-54 (Model 2).** Run this on Colab with a GPU runtime
(*Runtime → Change runtime type → T4 / L4 / A100*), then commit the outputs it writes.

Model 1 is the reference every other number is measured against. Model 2 is the first
model the team actually trains, which is what Team Meeting 2 checks on 12 November — a
prompted backbone does not satisfy that requirement.

### Before you start
You need the HinGE corpus. It is **not in the repository** (`data/raw/` is gitignored for
licence reasons), so have `HinGE.pkl` or `hinge.csv` ready to upload.

### What this writes, and what to commit
| Path | Commit? |
|---|---|
| `runs/*.json` | **yes** — the GPU-hour evidence |
| `results/results.jsonl` | **yes** — the scored comparison |
| `out/*.txt` | no — generated text, gitignored |
| `out/m2-seed*/` adapters | no — too large; keep the run record instead |
| `data/processed/` | no — gitignored |

The last cell zips exactly the files worth committing.

## 1. Check the GPU you were given

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, sys
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then run again.")
GPU = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n{GPU}  {VRAM:.0f} GB")

## 2. Pick a profile

Runtimes below are rough and measured against a 3,738-record train split and a
468-record test split.

| Profile | GPU | M1 (3 seeds, 0- and 5-shot) | M2 (1 seed) | Use when |
|---|---|---|---|---|
| `smoke` | any | ~3 min | ~5 min | proving the path end to end |
| `t4` | T4 16 GB | ~60 min | ~3 h, 1 epoch | free Colab |
| `a100` | A100 / L4 | ~20 min | ~1.5 h, 3 epochs | Colab Pro or the cluster |

**A full three-seed Model 2 is not realistic on a free T4.** Run `smoke` first to prove the
pipeline, then one seed on `t4`, and keep the remaining two seeds for the cluster once
298-51 confirms the allocation. Three seeds is what the success criterion requires, so a
one-seed result is a checkpoint demonstration, not the reported number — say so when you
present it.

In [ ]:
PROFILE = "smoke"          # "smoke" | "t4" | "a100"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # the adopted backbone (docs/workbook1/section_1_4.md)
SEEDS    = [13, 42, 1337]               # the project's fixed seeds

CFG = {
  "smoke": dict(model="Qwen/Qwen2.5-0.5B-Instruct", train_limit=256, max_steps=30,
                epochs=1, bsz=1, accum=16, maxlen=256, test_limit=48, seeds=[13]),
  "t4":    dict(model=MODEL_ID, train_limit=None, max_steps=None,
                epochs=1, bsz=1, accum=32, maxlen=320, test_limit=None, seeds=[13]),
  "a100":  dict(model=MODEL_ID, train_limit=None, max_steps=None,
                epochs=3, bsz=4, accum=8,  maxlen=512, test_limit=None, seeds=SEEDS),
}[PROFILE]

print(f"profile={PROFILE}")
for k, v in CFG.items():
    print(f"  {k:12s} {v}")
print(f"\n  effective batch = {CFG['bsz'] * CFG['accum']}")

## 3. Clone the repository and install dependencies

In [ ]:
import os, subprocess, pathlib
REPO = "https://github.com/298A-Team-2-Topic-23/Low-Resource-and-Code-Switched-Language-Systems.git"
ROOT = pathlib.Path("/content/lrcs")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
os.chdir(ROOT)
print("cwd:", os.getcwd())
print("HEAD:", subprocess.run(["git","rev-parse","--short","HEAD"],
                              capture_output=True, text=True).stdout.strip())

In [ ]:
%pip install -q "transformers>=4.51" "peft>=0.17" "bitsandbytes>=0.47" accelerate \
               datasketch sacrebleu sentencepiece
print("installed")

## 4. Get a corpus

Two ways to proceed.

**`USE_SYNTHETIC = True`** generates a HinGE-shaped corpus offline. Nothing to upload, and
it exercises the identical pipeline — this is the project's own documented fallback for the
demo. Use it for `smoke`, and whenever you just want to prove the path works.

**`USE_SYNTHETIC = False`** uses the real corpus, which you must upload: `HinGE.pkl`
(preferred — the conversion verifies its SHA-256 and unpickles through an allowlist) or an
already-converted `hinge.csv`.

The corpus is **not in the repository** — `data/raw/` is gitignored for licence reasons. If
you do not have it, ask Shibin or Yash, who have both run the real-corpus pipeline, or
fetch it from the Drive folder recorded in `docs/datasheet.md`.

**Any number produced from synthetic data is a pipeline demonstration, not a result.** The
notebook labels it as such throughout, and a model trained on it must never be reported as
a Model 2 score.

In [ ]:
USE_SYNTHETIC = True        # True = no upload needed; False = upload the real corpus

import pathlib
raw = pathlib.Path("data/raw"); raw.mkdir(parents=True, exist_ok=True)

if USE_SYNTHETIC:
    print("synthetic mode -- nothing to upload.")
    print("Numbers from this run demonstrate the pipeline; they are NOT results.")
else:
    from google.colab import files
    import shutil
    up = files.upload()                     # waits until you choose a file
    for name in up:
        shutil.move(name, raw / name)
        print("->", raw / name)
    assert (raw/"HinGE.pkl").exists() or (raw/"hinge.csv").exists(), \
        "need HinGE.pkl or hinge.csv in data/raw/ -- or set USE_SYNTHETIC = True"

## 5. Build the splits

On the real corpus the hashes printed below must match the ones committed in
`data/processed/manifests/hinge_split_manifest.json`. That match is the reproducibility
claim: same seed, same input hash, same splits byte for byte. If they differ, something
upstream changed and no model should be trained until it is explained.

On synthetic data there is nothing to compare against, so the check is skipped and the run
is marked as a demonstration.

In [ ]:
import subprocess, pathlib

if USE_SYNTHETIC:
    SOURCE = "synthetic"
    cmd = ["python","scripts/run_pipeline.py","--synthetic","--n-synthetic","1200",
           "--seed","42"]
else:
    SOURCE = "hinge"
    if not pathlib.Path("data/raw/hinge.csv").exists():
        subprocess.run(["python","scripts/convert_hinge_pkl.py",
                        "data/raw/HinGE.pkl","data/raw/hinge.csv"], check=True)
    cmd = ["python","scripts/run_pipeline.py","--input","data/raw/hinge.csv","--seed","42"]

r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2500:]); print(r.stderr[-800:])
assert r.returncode == 0, "pipeline failed -- do not train on this data"
print(f"\nsplits written under data/processed/{SOURCE}/")

In [ ]:
import json, hashlib, pathlib

if USE_SYNTHETIC:
    print("synthetic run -- no committed manifest to verify against.")
    print("Everything downstream is a pipeline demonstration, not a result.")
else:
    man = json.load(open("data/processed/manifests/hinge_split_manifest.json"))
    ok = True
    for name, entry in man["splits"].items():
        actual = hashlib.sha256(pathlib.Path(entry["file"]).read_bytes()).hexdigest()
        match = actual == entry["sha256"]
        ok &= match
        print(f"  {name:6s} {entry['records']:>5} records  {'MATCH' if match else 'DIFFERS'}")
    print("\nsplits reproduce byte for byte" if ok else
          "\nWARNING: splits differ from the committed manifest -- stop and investigate")

## 6. Prepare the evaluation files

The split TSVs carry `id, group, english, hindi, hinglish`. The generation script wants one
sentence per line, and the scorer wants hypotheses and references aligned by line number.

In [ ]:
import csv, pathlib
out = pathlib.Path("out"); out.mkdir(exist_ok=True)

def load(split):
    rows = list(csv.DictReader(open(f"data/processed/{SOURCE}/{split}.tsv"), delimiter="\t"))
    return rows[:CFG["test_limit"]] if (split == "test" and CFG["test_limit"]) else rows

test = load("test")
pathlib.Path("out/test.en.txt").write_text(
    "\n".join(r["english"] for r in test) + "\n", encoding="utf-8")
pathlib.Path("out/test.hinglish.txt").write_text(
    "\n".join(r["hinglish"] for r in test) + "\n", encoding="utf-8")

# few-shot examples must come from TRAIN, never dev or test
train = load("train")
with open("out/train.fewshot.tsv", "w", encoding="utf-8") as fh:
    for r in train:
        fh.write(f"{r['english']}\t{r['hinglish']}\n")

print(f"test {len(test)} lines | train {len(train)} rows")

## 7. Model 1 — zero-shot baseline (298-53)

0-shot and 5-shot are **separate conditions** and are reported separately, because the
success criterion is defined against this baseline and which one we mean has to be
unambiguous. Decoding is greedy so the baseline is deterministic.

In [ ]:
import subprocess, time
def run(cmd):
    print("$", " ".join(cmd))
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-1200:] or r.stderr[-1200:])
    print(f"[{time.time()-t0:.0f}s, exit {r.returncode}]\n")
    return r.returncode

for seed in CFG["seeds"]:
    run(["python","models/generate_zeroshot.py","--src","out/test.en.txt",
         "--out",f"out/m1.0shot.s{seed}.txt","--model",CFG["model"],
         "--shots","0","--seed",str(seed)])

In [ ]:
for seed in CFG["seeds"]:
    run(["python","models/generate_zeroshot.py","--src","out/test.en.txt",
         "--out",f"out/m1.5shot.s{seed}.txt","--model",CFG["model"],
         "--shots","5","--examples","out/train.fewshot.tsv","--seed",str(seed)])

In [ ]:
import pathlib
hyps0 = [f"out/m1.0shot.s{s}.txt" for s in CFG["seeds"] if pathlib.Path(f"out/m1.0shot.s{s}.txt").exists()]
hyps5 = [f"out/m1.5shot.s{s}.txt" for s in CFG["seeds"] if pathlib.Path(f"out/m1.5shot.s{s}.txt").exists()]
pathlib.Path("results").mkdir(exist_ok=True)
run(["python","evaluation/run_eval.py","--hyp",*hyps0,"--ref","out/test.hinglish.txt",
     "--system","Model 1 zero-shot (0-shot)","--out","results/results.jsonl"])
run(["python","evaluation/run_eval.py","--hyp",*hyps5,"--ref","out/test.hinglish.txt",
     "--system","Model 1 zero-shot (5-shot)","--out","results/results.jsonl"])

## 8. Model 2 — QLoRA fine-tune (298-54)

The recipe is fixed and must not be tuned here: rank 32, alpha 64, dropout 0.05, **all
linear projections**, 4-bit NF4 with double quantisation, loss masked to the completion.
M2 versus M3 later only isolates the vocabulary if this stays constant.

If this OOMs on a T4, lower `maxlen` or `bsz` in the profile — **do not** change rank,
alpha or the target modules.

In [ ]:
import subprocess, time
for seed in CFG["seeds"]:
    cmd = ["python","models/train_qlora.py",
           "--train",f"data/processed/{SOURCE}/train.tsv",
           "--src-col","2","--tgt-col","4",
           "--out",f"out/m2-seed{seed}","--model",CFG["model"],"--seed",str(seed),
           "--epochs",str(CFG["epochs"]),"--batch-size",str(CFG["bsz"]),
           "--grad-accum",str(CFG["accum"]),"--max-len",str(CFG["maxlen"])]
    if CFG["train_limit"]: cmd += ["--limit",str(CFG["train_limit"])]
    if CFG["max_steps"]:   cmd += ["--max-steps",str(CFG["max_steps"])]
    run(cmd)

In [ ]:
for seed in CFG["seeds"]:
    run(["python","models/generate_zeroshot.py","--src","out/test.en.txt",
         "--out",f"out/m2.s{seed}.txt","--model",CFG["model"],
         "--adapter",f"out/m2-seed{seed}","--shots","0","--seed",str(seed)])

hyps2 = [f"out/m2.s{s}.txt" for s in CFG["seeds"] if pathlib.Path(f"out/m2.s{s}.txt").exists()]
run(["python","evaluation/run_eval.py","--hyp",*hyps2,"--ref","out/test.hinglish.txt",
     "--system","Model 2 QLoRA","--out","results/results.jsonl"])

## 9. The comparison

The success criterion is **+3 chrF++ over zero-shot with non-overlapping error bars across
three seeds**. Below 1 chrF++, or overlapping bars, is a null result and gets published as
one — it does not get retried until it wins.

In [ ]:
run(["python","evaluation/run_eval.py","--report","results/results.jsonl"])

In [ ]:
import json, pathlib
rows = [json.loads(l) for l in pathlib.Path("results/results.jsonl").read_text().splitlines() if l.strip()]
base = next((r for r in rows if r["system"].startswith("Model 1") and "0-shot" in r["system"]), None)
m2   = next((r for r in rows if r["system"].startswith("Model 2")), None)
if base and m2:
    d = m2["metrics"]["chrf++"]["mean"] - base["metrics"]["chrf++"]["mean"]
    print(f"M1 0-shot chrF++ : {base['metrics']['chrf++']['mean']:.3f}")
    print(f"M2 QLoRA  chrF++ : {m2['metrics']['chrf++']['mean']:.3f}")
    print(f"delta            : {d:+.3f}")
    n = min(base["n_seeds"], m2["n_seeds"])
    if n < 3:
        print(f"\n{n} seed(s) only -- a checkpoint demonstration, not the reported "
              "number. The criterion needs three.")
    elif d >= 3.0: print("\nmeets the +3 chrF++ criterion -- check the error bars do not overlap")
    elif d < 1.0:  print("\nbelow 1 chrF++ -- this is a NULL RESULT and is published as one")
    else:          print("\nbetween 1 and 3 chrF++ -- inconclusive; report with error bars")
else:
    print("need both a Model 1 0-shot row and a Model 2 row")

## 10. Collect what to commit

Download `commit_me.zip`, unzip it over your clone, and push on a branch. Keep the
**`runs/*.json` and `results/results.jsonl`** — they are the evidence. The generated text
and the adapters stay out: `out/` is gitignored and the adapter weights are too large.

**Also record the GPU-hours** from the run records in `docs/COMPUTE_BUDGET.md`, in the same
commit as the run.

In [ ]:
import zipfile, pathlib, json
z = pathlib.Path("/content/commit_me.zip")
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(pathlib.Path("runs").glob("*.json")):
        zf.write(p, p.as_posix()); print("+", p)
    r = pathlib.Path("results/results.jsonl")
    if r.exists():
        zf.write(r, r.as_posix()); print("+", r)
print("\ntotal GPU-hours this session:")
tot = 0.0
for p in sorted(pathlib.Path("runs").glob("*.json")):
    d = json.loads(p.read_text())
    if "model" in d.get("name",""):
        tot += d.get("gpu_hours", 0)
        print(f"  {d['name']:<26} seed {d.get('seed')}  {d.get('gpu_hours',0):.3f} h")
print(f"  {'TOTAL':<26}        {tot:.3f} h")
from google.colab import files; files.download(str(z))